<a href="https://colab.research.google.com/github/e23046/Statistical-Learning-e23046/blob/main/Assignment%2005/Part__A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part A: Develop a Feature Analysis Dashboard

In [ ]:
# ============================================================
# FEATURE ANALYSIS DASHBOARD
# PCA Optimization Suite + FA Latent Subspace Suite
# ============================================================

!pip -q install factor_analyzer plotly

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from factor_analyzer import FactorAnalyzer, Rotator

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# MAIN FUNCTION
# ------------------------------------------------------------

def feature_analysis_dashboard(df, n_factors=2):

    # --------------------------------------------------------
    # 1. Standardization
    # --------------------------------------------------------
    X = df.select_dtypes(include=np.number)

    scaler = StandardScaler()
    Z = scaler.fit_transform(X)

    feature_names = X.columns.tolist()
    n_features = len(feature_names)

    # ========================================================
    # PCA SECTION
    # ========================================================

    pca = PCA()
    scores = pca.fit_transform(Z)

    eigenvalues = pca.explained_variance_
    explained_var = pca.explained_variance_ratio_ * 100
    cumulative_var = np.cumsum(explained_var)

    loadings = pca.components_.T

    mean_t2 = []
    mean_q = []

    for k in range(1, n_features + 1):

        Pk = loadings[:, :k]
        lambdak = eigenvalues[:k]

        Tk = Z @ Pk

        # Hotelling T²
        T2 = np.sum((Tk**2) / lambdak, axis=1)
        mean_t2.append(np.mean(T2))

        # Q Statistic (SPE)
        X_hat = Tk @ Pk.T
        residual = Z - X_hat

        Q = np.sum(residual**2, axis=1)
        mean_q.append(np.mean(Q))

    residual_var = 100 - cumulative_var

    # ========================================================
    # PCA DASHBOARD
    # ========================================================

    fig_pca = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=[
            "Feature Loadings Matrix",
            "Eigenvalues",
            "Explained vs Cumulative Variance",
            "Residual Variance",
            "Mean Hotelling T²",
            "Mean Q Statistic (SPE)"
        ]
    )

    # Panel 1,1 Heatmap
    fig_pca.add_trace(
        go.Heatmap(
            z=np.abs(loadings),
            x=[f"PC{i+1}" for i in range(n_features)],
            y=feature_names,
            colorscale="YlOrRd"
        ),
        row=1,
        col=1
    )

    # Panel 1,2 Eigenvalues
    fig_pca.add_trace(
        go.Bar(
            x=[f"PC{i+1}" for i in range(n_features)],
            y=np.abs(eigenvalues),
            name="Eigenvalues"
        ),
        row=1,
        col=2
    )

    # Panel 1,3 Explained + Cumulative
    fig_pca.add_trace(
        go.Bar(
            x=[f"PC{i+1}" for i in range(n_features)],
            y=explained_var,
            name="Explained %"
        ),
        row=1,
        col=3
    )

    fig_pca.add_trace(
        go.Scatter(
            x=[f"PC{i+1}" for i in range(n_features)],
            y=cumulative_var,
            mode="lines+markers",
            line=dict(dash="dash"),
            name="Cumulative %"
        ),
        row=1,
        col=3
    )

    # Panel 2,1 Residual Variance
    fig_pca.add_trace(
        go.Bar(
            x=[f"k={i+1}" for i in range(n_features)],
            y=residual_var,
            name="Residual %"
        ),
        row=2,
        col=1
    )

    # Panel 2,2 Mean T²
    fig_pca.add_trace(
        go.Scatter(
            x=list(range(1, n_features + 1)),
            y=mean_t2,
            mode="lines+markers",
            name="Mean T²"
        ),
        row=2,
        col=2
    )

    # Panel 2,3 Mean Q
    fig_pca.add_trace(
        go.Scatter(
            x=list(range(1, n_features + 1)),
            y=mean_q,
            mode="lines+markers",
            name="Mean Q"
        ),
        row=2,
        col=3
    )

    fig_pca.update_layout(
        title="PCA Optimization Suite",
        height=750,
        width=1400,
        template="plotly_white",
        showlegend=True
    )

    fig_pca.show()

    # ========================================================
    # FACTOR ANALYSIS SECTION
    # ========================================================

    fa = FactorAnalyzer(
        n_factors=n_factors,
        rotation=None,
        method="ml"
    )

    fa.fit(Z)

    loadings_fa = fa.loadings_

    rotator = Rotator(method="varimax")
    rotated_loadings = rotator.fit_transform(loadings_fa)

    communalities = np.sum(rotated_loadings**2, axis=1)
    uniqueness = 1 - communalities

    R = np.corrcoef(Z.T)

    factor_scores = Z @ np.linalg.inv(R) @ rotated_loadings

    factor_variance = np.var(
        factor_scores,
        axis=0,
        ddof=1
    )

    # ========================================================
    # FA DASHBOARD
    # ========================================================

    fig_fa = make_subplots(
        rows=2,
        cols=2,
        horizontal_spacing=0.24,
        vertical_spacing=0.28,
        subplot_titles=[
            "Structural Loadings Matrix",
            "Variance Allocation",
            "Sensor Uniqueness Profile",
            "Latent Factor Variance"
        ]
    )

    # Panel 1,1
    fig_fa.add_trace(
        go.Heatmap(
            z=np.abs(rotated_loadings),
            x=[f"Factor {i+1}" for i in range(n_factors)],
            y=feature_names,
            colorscale="YlOrRd",
            colorbar=dict(x=-0.15)
        ),
        row=1,
        col=1
    )

    # Panel 1,2
    fig_fa.add_trace(
        go.Bar(
            y=feature_names,
            x=communalities * 100,
            orientation="h",
            name="Communality h²"
        ),
        row=1,
        col=2
    )

    fig_fa.add_trace(
        go.Bar(
            y=feature_names,
            x=uniqueness * 100,
            orientation="h",
            name="Uniqueness φ²"
        ),
        row=1,
        col=2
    )

    # Panel 2,1
    fig_fa.add_trace(
        go.Scatter(
            x=feature_names,
            y=uniqueness,
            mode="lines+markers",
            name="φ² Profile"
        ),
        row=2,
        col=1
    )

    # Panel 2,2
    fig_fa.add_trace(
        go.Bar(
            x=[f"Factor {i+1}" for i in range(n_factors)],
            y=factor_variance,
            name="Factor Variance"
        ),
        row=2,
        col=2
    )

    fig_fa.update_layout(
        barmode="stack",
        template="plotly_white",
        width=1250,
        height=750,
        margin=dict(
            t=150,
            b=60,
            l=140,
            r=80
        ),
        legend=dict(
            orientation="h",
            x=0.5,
            y=1.02,
            xanchor="center"
        ),
        title="FA Latent Subspace Suite"
    )

    fig_fa.show()

    # --------------------------------------------------------
    # Console Summary
    # --------------------------------------------------------

    print("="*60)
    print("FA DIAGNOSTIC SUMMARY")
    print("="*60)

    print(
        f"Average System Communality (%) : "
        f"{communalities.mean()*100:.2f}"
    )

    print(
        f"Average System Uniqueness (%) : "
        f"{uniqueness.mean()*100:.2f}"
    )

    return {
        "pca": pca,
        "fa_loadings": rotated_loadings,
        "communality": communalities,
        "uniqueness": uniqueness,
        "factor_variance": factor_variance
    }

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
np.random.seed(42)
sample_data = np.random.rand(100, 10)
feature_names_sample = [f'feature_{i+1}' for i in range(10)]
df_asset = pd.DataFrame(sample_data, columns=feature_names_sample)
display(df_asset.head())

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10
0,0.374540,0.950714,0.731994,0.598658,0.156019,0.155995,0.058084,0.866176,0.601115,0.708073
1,0.020584,0.969910,0.832443,0.212339,0.181825,0.183405,0.304242,0.524756,0.431945,0.291229
2,0.611853,0.139494,0.292145,0.366362,0.456070,0.785176,0.199674,0.514234,0.592415,0.046450
3,0.607545,0.170524,0.065052,0.948886,0.965632,0.808397,0.304614,0.097672,0.684233,0.440152
4,0.122038,0.495177,0.034389,0.909320,0.258780,0.662522,0.311711,0.520068,0.546710,0.184854


In [ ]:
results = feature_analysis_dashboard(
    df_asset,
    n_factors=2
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



FA DIAGNOSTIC SUMMARY
Average System Communality (%) : 17.76
Average System Uniqueness (%) : 82.24
